# JUMP cp_measure feature prediction

Per-feature ridge regression `embedding -> cp_measure features`. For each
(encoder, view, embedding) triple we fit on non-held-out wells and report
two metrics on the held-out source:

- **`r2`** — coefficient of determination per feature. Captures *all*
  explainable variance, including source-level offsets that leak through
  Harmony residue. Reported for completeness; will overstate
  perturbation-driven signal when the embedding still carries
  source-discriminative information.
- **`pearson_r`** — Pearson correlation between predicted and true
  cp_measure values across held-out-source wells, per feature.
  Naturally offset-invariant (subtracting per-source means changes
  nothing) so it isolates *perturbation-driven* covariation between
  the embedding's predictions and ground truth.

The gap `R² - r²` (where `r` is Pearson) is the source-confound
contribution. Headline metric: `median_pearson_r` and
`frac_pearson_r_gt_0p3` (across cp_measure features).

Held-out batch: `source_4` (consistent with recall + copairs evals).

Predictor: ridge (α=1.0). Linear probe is the canonical comparison
across encoders. Nonlinear baselines (multi-output XGBoost,
polynomial-features ridge) were attempted and dropped — see git
history for details.

In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
EMBED_DIR = Path('../data')
CP_PATH = Path('_data') / 'jump_cp_well.h5ad'

DATASET = 'JUMP'
EMBEDDING_KEYS = ('X_pca', 'X_pca_harmony')
JOIN_KEYS = ('source', 'plate', 'well')
HELD_OUT_BATCH = 'source_4'

RIDGE_ALPHA = 1.0

GROUPS: dict[str, dict[str, str]] = {
    'JUMP — DINO': {
        'C':  '01_JUMP_DINO_ECFP4_C_aggregated.h5ad',
        'CD': '02_JUMP_DINO_ECFP4_CD_aggregated.h5ad',
        'S':  '03_JUMP_DINO_ECFP4_S_aggregated.h5ad',
        'SD': '04_JUMP_DINO_ECFP4_SD_aggregated.h5ad',
    },
    'JUMP — OpenPhenom': {
        'C':  '05_JUMP_OpenPhenom_ECFP4_C_aggregated.h5ad',
        'CD': '06_JUMP_OpenPhenom_ECFP4_CD_aggregated.h5ad',
        'S':  '07_JUMP_OpenPhenom_ECFP4_S_aggregated.h5ad',
        'SD': '08_JUMP_OpenPhenom_ECFP4_SD_aggregated.h5ad',
        'zero-shot': 'OpenPhenom_zeroshot_JUMP_aggregated.h5ad',
    },
    'JUMP — SubCell': {
        'C':  '09_JUMP_SubCell_ECFP4_C_aggregated.h5ad',
        'CD': '10_JUMP_SubCell_ECFP4_CD_aggregated.h5ad',
        'S':  '11_JUMP_SubCell_ECFP4_S_aggregated.h5ad',
        'SD': '12_JUMP_SubCell_ECFP4_SD_aggregated.h5ad',
    },
}

In [ ]:
# cp_measure feature names: '<Compartment>_<Family>_<...>_<Channel>'.
# Map second token to one of seven semantic groups.
TEXTURE_FAMILIES = {
    'AngularSecondMoment', 'Contrast', 'Correlation', 'DifferenceEntropy',
    'DifferenceVariance', 'Entropy', 'InfoMeas1', 'InfoMeas2',
    'InverseDifferenceMoment', 'SumAverage', 'SumEntropy', 'SumVariance',
    'Variance',
}
AREASHAPE_FAMILIES = {
    'BoundingBoxMaximum', 'BoundingBoxMinimum', 'Center', 'CentralMoment',
    'Compactness', 'ConvexArea', 'Eccentricity', 'EquivalentDiameter',
    'EulerNumber', 'Extent', 'FormFactor', 'HuMoment', 'InertiaTensor',
    'InertiaTensorEigenvalues', 'Location', 'MajorAxisLength',
    'MaximumRadius', 'MedianRadius', 'MinFeretDiameter', 'MinorAxisLength',
    'NormalizedMoment', 'Orientation', 'Solidity', 'SpatialMoment',
}
GROUP_ORDER = ('Intensity', 'Texture', 'Granularity', 'RadialDistribution',
               'AreaShape', 'Zernike', 'Other')

def feature_group(name: str) -> str:
    parts = name.split('_')
    family = parts[1] if len(parts) > 1 else 'Other'
    if family == 'Intensity':
        return 'Intensity'
    if family == 'Granularity':
        return 'Granularity'
    if family == 'RadialDistribution':
        return 'RadialDistribution'
    if family == 'Zernike':
        return 'Zernike'
    if family in TEXTURE_FAMILIES:
        return 'Texture'
    if family in AREASHAPE_FAMILIES:
        return 'AreaShape'
    return 'Other'

In [ ]:
# Load CP target once. Features are encoder-independent; we'll reuse this
# array for every config by joining on (source, plate, well).
cp = ad.read_h5ad(CP_PATH)
cp_obs = cp.obs[list(JOIN_KEYS)].astype(str).reset_index(drop=True)
cp_X = np.asarray(cp.X, dtype=np.float32)
cp_var = np.array(cp.var_names)
cp_groups = np.array([feature_group(n) for n in cp_var])

print(f'CP target: {cp_X.shape}, groups: {dict(zip(*np.unique(cp_groups, return_counts=True)))}')

cp_idx_map = pd.Series(
    np.arange(len(cp_obs)),
    index=pd.MultiIndex.from_arrays(
        [cp_obs[k].to_numpy() for k in JOIN_KEYS], names=list(JOIN_KEYS)
    ),
)

In [ ]:
def align(emb: ad.AnnData, embedding_key: str) -> tuple[np.ndarray, np.ndarray, pd.DataFrame, pd.MultiIndex]:
    emb_obs = emb.obs[list(JOIN_KEYS) + ['batch', 'is_control']].astype(str).reset_index(drop=True)
    emb_key = pd.MultiIndex.from_arrays(
        [emb_obs[k].to_numpy() for k in JOIN_KEYS], names=list(JOIN_KEYS)
    )
    shared_mask = emb_key.isin(cp_idx_map.index)
    if not shared_mask.all():
        emb_obs = emb_obs[shared_mask].reset_index(drop=True)
        emb_key = emb_key[shared_mask]
    X = np.asarray(emb.obsm[embedding_key], dtype=np.float32)[shared_mask]
    Y = cp_X[cp_idx_map.loc[emb_key].to_numpy()]
    return X, Y, emb_obs, emb_key

def per_feature_pearson(Y_pred: np.ndarray, Y_true: np.ndarray) -> np.ndarray:
    """Pearson r per output column, vectorised."""
    yp = Y_pred - Y_pred.mean(axis=0, keepdims=True)
    yt = Y_true - Y_true.mean(axis=0, keepdims=True)
    num = (yp * yt).sum(axis=0)
    den = np.sqrt((yp ** 2).sum(axis=0) * (yt ** 2).sum(axis=0))
    out = np.full(Y_pred.shape[1], np.nan, dtype=np.float64)
    nz = den > 0
    out[nz] = num[nz] / den[nz]
    return out

def summarise(metric_arr: np.ndarray, kept_groups: np.ndarray, prefix: str) -> dict:
    finite = np.isfinite(metric_arr)
    arr = metric_arr[finite]
    grp = kept_groups[finite]
    out = {
        f'{prefix}_n': int(len(arr)),
        f'{prefix}_median': float(np.median(arr)) if len(arr) else float('nan'),
        f'{prefix}_iqr_lo': float(np.quantile(arr, 0.25)) if len(arr) else float('nan'),
        f'{prefix}_iqr_hi': float(np.quantile(arr, 0.75)) if len(arr) else float('nan'),
        f'{prefix}_frac_gt_0': float((arr > 0).mean()) if len(arr) else float('nan'),
        f'{prefix}_frac_gt_0p1': float((arr > 0.1).mean()) if len(arr) else float('nan'),
        f'{prefix}_frac_gt_0p3': float((arr > 0.3).mean()) if len(arr) else float('nan'),
    }
    for g in GROUP_ORDER:
        m = grp == g
        out[f'{prefix}_{g.lower()}'] = float(np.median(arr[m])) if m.any() else float('nan')
    return out

def evaluate(emb: ad.AnnData, embedding_key: str) -> dict:
    X, Y, obs, _ = align(emb, embedding_key)
    is_control = obs['is_control'].values == 'True'
    is_held_out = obs['batch'].values == HELD_OUT_BATCH
    train_mask = (~is_held_out) & (~is_control)
    test_mask = is_held_out & (~is_control)

    Y_train_full = Y[train_mask]
    keep = Y_train_full.var(axis=0) > 1e-8
    Y_train = Y_train_full[:, keep]
    Y_test = Y[test_mask][:, keep]
    kept_groups = cp_groups[keep]

    scaler = StandardScaler().fit(X[train_mask])
    X_train = scaler.transform(X[train_mask])
    X_test = scaler.transform(X[test_mask])

    t0 = time.time()
    ridge = Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_SEED)
    ridge.fit(X_train, Y_train)
    Y_pred = ridge.predict(X_test)
    fit_seconds = round(time.time() - t0, 1)

    r2 = r2_score(Y_test, Y_pred, multioutput='raw_values')
    pearson = per_feature_pearson(Y_pred, Y_test)

    row = {'n_train': int(train_mask.sum()), 'n_test': int(test_mask.sum()),
           'fit_seconds': fit_seconds}
    row.update(summarise(r2, kept_groups, 'r2'))
    row.update(summarise(pearson, kept_groups, 'pearson'))
    return row

In [ ]:
summary_rows = []
for group_name, view_files in GROUPS.items():
    print(f'=== {group_name} ===')
    for view, fname in view_files.items():
        path = EMBED_DIR / fname
        emb = ad.read_h5ad(path)
        for emb_key in EMBEDDING_KEYS:
            tag = 'pre_harmony' if emb_key == 'X_pca' else 'post_harmony'
            row = evaluate(emb, emb_key)
            row.update({'group': group_name, 'view': view, 'embedding': tag})
            summary_rows.append(row)
            print(f'  {view:>9s} {tag:>13s}  '
                  f'R² med={row["r2_median"]:+.3f} >0.1={row["r2_frac_gt_0p1"]:.2f}  '
                  f'r med={row["pearson_median"]:+.3f} >0.3={row["pearson_frac_gt_0p3"]:.2f}  '
                  f'({row["fit_seconds"]:.1f}s)')

summary = pd.DataFrame(summary_rows)
first_cols = ['group', 'view', 'embedding']
summary = summary[first_cols + [c for c in summary.columns if c not in first_cols]]
summary

In [ ]:
summary.to_csv(f'cp_feature_prediction_{DATASET.lower()}.csv', index=False)
summary.pivot_table(
    index=['group', 'view'], columns='embedding',
    values=['r2_median', 'pearson_median', 'pearson_frac_gt_0p3'],
)

In [ ]:
VIEW_STYLES = {
    'C':         {'color': 'C0', 'hatch': None,    'alpha': 1.00},
    'CD':        {'color': 'C1', 'hatch': None,    'alpha': 1.00},
    'S':         {'color': 'C2', 'hatch': None,    'alpha': 1.00},
    'SD':        {'color': 'C3', 'hatch': None,    'alpha': 1.00},
    'zero-shot': {'color': 'gray', 'hatch': '//',  'alpha': 0.85},
}
VIEW_ORDER = ['C', 'CD', 'S', 'SD', 'zero-shot']

def plot_metric(metric_col: str, ylabel: str, ymax: float | None, fname: str) -> None:
    n_groups = len(GROUPS)
    for emb in EMBEDDING_KEYS:
        tag = 'pre_harmony' if emb == 'X_pca' else 'post_harmony'
        fig, axes = plt.subplots(1, n_groups, figsize=(4.5 * n_groups, 4),
                                 sharey=True, squeeze=False)
        for ax, (group_name, view_files) in zip(axes[0], GROUPS.items()):
            views = [v for v in VIEW_ORDER if v in view_files]
            xs = np.arange(len(views))
            sub = summary[(summary['group'] == group_name) & (summary['embedding'] == tag)].set_index('view')
            ys = [sub.loc[v, metric_col] for v in views]
            for x, v, y in zip(xs, views, ys):
                style = VIEW_STYLES[v]
                ax.bar(x, y, color=style['color'], hatch=style['hatch'],
                       alpha=style['alpha'], edgecolor='black', linewidth=0.7)
                ax.text(x, y + (0.005 if ymax is None else 0.01),
                        f'{y:.2f}', ha='center', va='bottom', fontsize=8)
            ax.set_xticks(xs)
            ax.set_xticklabels(views, rotation=0)
            ax.set_title(group_name.replace(f'{DATASET} — ', ''))
            ax.set_xlabel('view')
            ax.grid(axis='y', alpha=0.3)
        axes[0, 0].set_ylabel(ylabel)
        if ymax is not None:
            axes[0, 0].set_ylim(0, ymax)
        fig.suptitle(f'{DATASET} {ylabel} ({tag})', y=1.02)
        fig.tight_layout()
        out = f'{fname}_{DATASET.lower()}_{tag}.png'
        fig.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  saved {out}')

plot_metric('pearson_median', 'median Pearson r', None, 'cp_feature_prediction_pearson_median')
plot_metric('pearson_frac_gt_0p3', 'fraction Pearson r > 0.3', 1.05, 'cp_feature_prediction_pearson_sigfrac')
plot_metric('r2_median', 'median R² (uncalibrated)', None, 'cp_feature_prediction_r2_median')

In [ ]:
import seaborn as sns

GROUP_COLS = [f'pearson_{g.lower()}' for g in GROUP_ORDER]
GROUP_LABELS = list(GROUP_ORDER)

for emb in EMBEDDING_KEYS:
    tag = 'pre_harmony' if emb == 'X_pca' else 'post_harmony'
    df = summary[summary['embedding'] == tag].copy()
    df['row_label'] = df['group'].str.replace(f'{DATASET} — ', '') + ' · ' + df['view']
    mat = df.set_index('row_label')[GROUP_COLS].to_numpy()
    fig, ax = plt.subplots(figsize=(1.0 + 0.7 * len(GROUP_LABELS), 0.4 * len(df)))
    sns.heatmap(mat, cmap='RdBu_r', center=0, vmin=-0.2, vmax=0.6,
                xticklabels=GROUP_LABELS, yticklabels=df['row_label'].tolist(),
                annot=True, fmt='.2f', annot_kws={'fontsize': 7}, ax=ax,
                cbar_kws={'label': 'median Pearson r (per group)'})
    ax.set_title(f'{DATASET} median Pearson r by feature group ({tag})')
    fig.tight_layout()
    out = f'cp_feature_prediction_groups_{DATASET.lower()}_{tag}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  saved {out}')